# Dark-Photon FIRAS Limits: Paper Fig. 8 with the CCJ24 Statistic Overlaid

Paper Fig. 8 (`dp_firas_pde_constraints.pdf`, generated by
`notebooks/paper_figures/dark_photon_constraints.ipynb`) shows the FIRAS 95% CL
upper limit on the kinetic mixing $\epsilon(m_{A'})$ from `spectroxide` PDE
templates, fit with a **floating-$T$ profile likelihood** and the full
$43\times43$ FIRAS covariance, against the digitized published
`CosmoTherm` limit of
[Chluba, Cyr &amp; Johnson (2024)](https://arxiv.org/abs/2409.13818) (CCJ24).

This notebook reproduces that figure — same self-consistent
$\epsilon_{\rm ref}$ iteration, same $\gamma_{\rm con} < 0.1$ cutoff, same
styling, with the mass grid truncated at $1.4\times10^{-4}$ eV (above which no
point survives the cut) and the solver run at the tighter
`dtau_max = 3` — and adds **one extra curve**: the same PDE templates run
through the **CCJ24 statistic** (diagonal covariance, fixed-$T_0$ linear
deprojection of the blackbody and galactic-dust templates, one-sided
$\Delta\chi^2 = 2.71$ likelihood interval).

The two `spectroxide` curves come from **the same PDE templates** — the
statistic is only a fit applied afterwards, so no extra solver runs are needed.
The only thing that differs between them is the statistic, not the solver, the
template, or the cutoff.

Output: `notebooks/figures/dp_firas_pde_constraints_ccj24.pdf`

In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# Ensure cargo is on PATH for the Rust solver binary
cargo_bin = Path.home() / '.cargo' / 'bin'
if cargo_bin.is_dir() and str(cargo_bin) not in os.environ.get('PATH', ''):
    os.environ['PATH'] = str(cargo_bin) + os.pathsep + os.environ.get('PATH', '')

import numpy as np
import matplotlib.pyplot as plt

from spectroxide import apply_style, g_bb, mu_shape
from spectroxide.dark_photon import gc_per_epsilon_sq, resonance_redshift
from spectroxide.firas import FIRASData, _dn_to_dI_kJy
from spectroxide.plot_params import C, DOUBLE_COL, LW_THICK, LEGEND_SIZE
from spectroxide.solver import solve

apply_style()

PROJECT_ROOT = Path.cwd().parent.parent
FIG_DIR = PROJECT_ROOT / 'notebooks' / 'figures'
DATA_DIR = PROJECT_ROOT / 'dev' / 'data'
CACHE = DATA_DIR / 'dp_ccj24_overlay.npz'

firas = FIRASData()
print(firas)

## The two statistics

**Profile likelihood (paper Fig. 8).** The model
$I_{\rm obs}(\nu) - B(\nu, T) = 4\pi\nu^3\,\gamma_{\rm con}\,\mathcal{T}(x(T))
+ G_0\,\nu^2 B(\nu, T_d)$ is fit with the full FIRAS covariance; $T$ is profiled
numerically, $\gamma_{\rm con}$ and $G_0$ analytically. The one-sided 95% CL
limit is where the profile likelihood ratio crosses $\lambda = 2.71$.

**CCJ24 statistic.** Diagonal errors only; the blackbody-shift and
galactic-dust templates are linearly projected out at fixed $T_0$; the limit is
$\hat a + \sqrt{2.71}\,\sigma_a$ on the remaining amplitude, with $\hat a$ floored
at zero.

In [ ]:
def _deprojected_model(template_dn_func):
    """CCJ24 deprojection: model in kJy/sr, T-shift + dust removed (diag C)."""
    S = _dn_to_dI_kJy(firas.x, template_dn_func(firas.x))
    G = firas.gbb_template_kJy()
    g = firas.galactic_template_kJy()
    w = 1.0 / firas.sigma_kJy**2

    def dot(a, b):
        return float(np.sum(a * w * b))

    m2 = np.array([[dot(G, G), dot(g, G)], [dot(G, g), dot(g, g)]])
    dT, dg = np.linalg.solve(m2, [dot(S, G), dot(S, g)])
    return S - dT * G - dg * g, dot


def ccj24_limit(template_dn_func, cl=0.95):
    """CCJ24-style limit: diagonal covariance, fixed-T0 deprojection,
    one-sided upper limit at Delta chi^2 = 2.71 above the (non-negative)
    best-fit amplitude."""
    S_perp, dot = _deprojected_model(template_dn_func)
    d = firas.residual_kJy
    A = dot(S_perp, S_perp)
    B = dot(d, S_perp)
    a_hat = max(B / A, 0.0)
    return a_hat + np.sqrt(2.71) / np.sqrt(A)


def profile_limit(template_dn_func, cl=0.95):
    """Floating-T profile-likelihood limit (paper Fig. 8 method)."""
    return firas.profile_limit_floating_T(template_dn_func, cl=cl)['upper_limit']


# Sanity check on a pure mu-distortion template
print(f'|mu| < {ccj24_limit(mu_shape):.2e}   (CCJ24 statistic)')
print(f'|mu| < {profile_limit(mu_shape):.2e}   (profile likelihood, paper Fig. 8)')

## PDE templates and the self-consistent $\epsilon_{\rm ref}$ iteration

One PDE run per dark-photon mass. The solver installs the impulsive depletion
$\Delta n(x, z_{\rm res}) = -[1 - e^{-\gamma_{\rm con}/x}]\,n_{\rm pl}(x)$ at
$z_{\rm res}$ and evolves to $z = 100$; the number-conserving
$G_{\rm bb}$-stripped output, divided by $\gamma_{\rm con,ref}$, is the template
$\mathcal{T}(x; m_{A'})$.

Because the depletion saturates for $\gamma_{\rm con} \gtrsim x$, the template
shape depends on the reference mixing used to generate it. The reference is
therefore iterated to self-consistency with the limit it produces, with the
low-$\epsilon$/high-$\epsilon$ 2-cycle resolved onto the conservative
(linear-NWA) branch. This is the paper notebook's `_pde_worker`, unchanged.

The iteration is driven by the profile likelihood, i.e. the paper's own
$\epsilon_{\rm ref}$; the converged template is then reused for both fits.
Strictly the self-consistent point is statistic-dependent, but the two limits
differ by only $\epsilon_{\rm ccj}/\epsilon_{\rm pl} = 1.16$, a $35\%$ shift in
$\gamma_{\rm con}$ that changes the template only where the depletion
saturates — the region already removed by the $\gamma_{\rm con} < 0.1$ cut.

In [ ]:
def strip_gbb_nc(x, delta_n):
    """Number-conserving G_bb subtraction: int x^2 Dn_stripped dx = 0."""
    gbb = g_bb(x)
    alpha = np.trapz(x**2 * delta_n, x) / np.trapz(x**2 * gbb, x)
    return delta_n - alpha * gbb, alpha


# Tighter than the solver default dtau_max=10 used for paper Fig. 8:
# the CLI documents dtau_max=3 as the <0.1% setting, and dtau_max is the
# only knob that moves the residual energy-conservation deficit.
DTAU_MAX = 3.0


def run_dp_pde(epsilon, m_ev, npts):
    result = solve(
        injection={'type': 'dark_photon_resonance',
                   'epsilon': epsilon, 'm_ev': m_ev},
        z_end=100, n_points=npts, dtau_max=DTAU_MAX, timeout=5400,
        # 5400 s: at dtau_max=3 the m ~ 1e-4 eV chains (z_res ~ 3e6)
        # exceed the usual 1200 s cap under 10-way CPU contention.
    )
    return result.x, result.delta_n


def _pde_single_pass(m_dp, eps_ref, npts, limit_fn):
    zr = resonance_redshift(m_dp)
    if zr is None:
        return None, None, None, None
    gpe2, _ = gc_per_epsilon_sq(m_dp)
    gc_ref = gpe2 * eps_ref**2
    x_pde, dn_pde = run_dp_pde(eps_ref, m_dp, npts)
    dn_nc, _ = strip_gbb_nc(x_pde, dn_pde)
    dn_per_gc = dn_nc / gc_ref

    def template(x, _x=x_pde, _dn=dn_per_gc):
        return np.interp(x, _x, _dn)

    gc_95 = limit_fn(template)
    eps_lim = np.sqrt(gc_95 / gpe2) if gc_95 > 0 and gpe2 > 0 else np.inf
    return eps_lim, x_pde, dn_per_gc, zr


def _pde_worker(m_dp, eps_ref_init, limit_fn=None,
                extrap_trigger=3.0, rtol=0.05, max_iter=10):
    """Self-consistent eps_ref iteration for a single mass.

    Accept a single pass when the limit extrapolates only mildly; otherwise
    iterate eps_ref <- eps_lim. The update can land on a 2-cycle (low-eps
    linear branch vs high-eps nonlinear-saturated branch); detect it by
    comparing iterate i with i-2 and return the HIGH-eps (conservative,
    linear-NWA) branch, matching CCJ24's methodology.
    """
    zr = resonance_redshift(m_dp)
    if zr is None:
        return m_dp, None, None, None, None, 0, False
    npts = 4000 if zr < 1e6 else 8000

    history = []
    eps_ref = eps_ref_init
    converged = False
    try:
        for it in range(max_iter):
            eps_lim, x_pde, dn_per_gc, _ = _pde_single_pass(
                m_dp, eps_ref, npts, limit_fn)
            if eps_lim is None or not np.isfinite(eps_lim):
                break
            history.append((eps_ref, eps_lim, x_pde, dn_per_gc))

            if it > 0:
                prev = history[-2][1]
                if abs(eps_lim - prev) / prev < rtol:
                    converged = True
                    break
            if it > 1:
                two_back = history[-3][1]
                if abs(eps_lim - two_back) / two_back < rtol:
                    # keep the HIGH-eps (conservative, linear-NWA) branch
                    hi = -1 if eps_lim > prev else -2
                    _, eps_lim, x_pde, dn_per_gc = history[hi]
                    converged = True
                    break
            if it == 0 and eps_lim / eps_ref < extrap_trigger:
                converged = True
                break
            eps_ref = eps_lim

        if not history:
            return m_dp, zr, None, None, None, 0, False
        return (m_dp, zr, eps_lim, x_pde, dn_per_gc, len(history), converged)
    except Exception as exc:
        print(f'  m={m_dp:.2e} eV FAILED: {exc}', file=sys.stderr)
        return m_dp, zr, None, None, None, len(history), False


def build_templates(masses, limit_fn=profile_limit, eps_ref_init=1e-8,
                    workers=10):
    """One PDE chain per mass. Returns {mass: {...}} including the template."""
    out = {}
    print(f'launching {len(masses)} PDE chains...', flush=True)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(_pde_worker, m, eps_ref_init, limit_fn): m
                   for m in masses}
        for k, fut in enumerate(as_completed(futures), 1):
            m_dp, zr, eps_lim, x_pde, dn_per_gc, n_iter, conv = fut.result()
            if eps_lim is None or zr is None:
                print(f'  {k}/{len(masses)} m={m_dp:.2e} eV: no limit',
                      flush=True)
                continue
            gpe2, _ = gc_per_epsilon_sq(m_dp)
            out[m_dp] = {'z_res': zr, 'gpe2': gpe2, 'eps_pl': eps_lim,
                         'x': x_pde, 'dn_per_gc': dn_per_gc,
                         'converged': conv, 'n_iter': n_iter}
            flag = '' if conv else '  !! NOT CONVERGED'
            print(f'  {k}/{len(masses)} m={m_dp:.2e} eV z_res={zr:.3g} '
                  f'eps={eps_lim:.3e} [{n_iter} iter]{flag}', flush=True)
    n_bad = sum(not v['converged'] for v in out.values())
    print(f'completed {len(out)}/{len(masses)}, {n_bad} not converged',
          flush=True)
    return out


def apply_statistic(templates, limit_fn):
    """Re-fit the *same* templates with a different statistic. No PDE runs."""
    eps = {}
    for m, t in templates.items():
        def tmpl(xx, _x=t['x'], _dn=t['dn_per_gc']):
            return np.interp(xx, _x, _dn)
        eps[m] = np.sqrt(limit_fn(tmpl) / t['gpe2'])
    return eps

## Run the sweep and apply both statistics

76 PDE chains, a few solver calls each; the second statistic is then a pure
re-fit of the templates already in memory. Set `USE_CACHE = False` to force a
fresh run instead of reloading `dev/data/dp_ccj24_overlay.npz`.

In [ ]:
USE_CACHE = True   # set False to force a fresh run

# Paper mass grid (cell 14 of paper_figures/dark_photon_constraints.ipynb)
masses_pde = np.unique(np.concatenate([
    np.geomspace(1e-12, 2e-9, 20),
    np.geomspace(3e-9, 1e-7, 30),
    np.geomspace(2e-7, 5e-5, 12),
    np.geomspace(6e-5, 1.427e-4, 8),
]))
# The paper grid runs to 3e-4 eV, but every mass above 1.5e-4 eV lands on a
# bistable eps_ref 2-cycle that never converges and is removed by the
# gamma_con < 0.1 cut anyway — at ~10 solver calls each they were most of the
# runtime and none of the figure.

if USE_CACHE and CACHE.exists():
    d = dict(np.load(CACHE))
    print(f'loaded cache {CACHE}')
else:
    templates = build_templates(masses_pde)          # the only PDE cost
    eps_ccj_map = apply_statistic(templates, ccj24_limit)   # pure re-fit

    ms = np.array(sorted(templates))
    e_pl = np.array([templates[m]['eps_pl'] for m in ms])
    e_cj = np.array([eps_ccj_map[m] for m in ms])
    gpe2 = np.array([templates[m]['gpe2'] for m in ms])
    d = dict(m=ms, e_pl=e_pl, e_cj=e_cj,
             gc_pl=gpe2 * e_pl**2, gc_cj=gpe2 * e_cj**2,
             cv=np.array([templates[m]['converged'] for m in ms]))
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez(CACHE, **d)
    print(f'cached -> {CACHE}')

## Figure

Identical to paper Fig. 8 apart from the added dash-dotted curve. Both
`spectroxide` curves are cut at $\gamma_{\rm con} < 0.1$ (beyond which the
linear-NWA template extrapolation breaks down) and drop masses whose
$\epsilon_{\rm ref}$ iteration did not converge.

In [ ]:
cosmotherm_lims = np.loadtxt(DATA_DIR / 'cosmotherm_dp_lims.csv', delimiter=',')


def valid(m, e, gc, cv):
    mask = (gc < 0.1) & cv.astype(bool)
    return m[mask], e[mask]


m_pl_v, e_pl_v = valid(d['m'], d['e_pl'], d['gc_pl'], d['cv'])
m_cj_v, e_cj_v = valid(d['m'], d['e_cj'], d['gc_cj'], d['cv'])

# --- Existing laboratory limits (non-DM, non-FIRAS) from AxionLimits ---
AL = PROJECT_ROOT / 'dev' / 'AxionLimits' / 'limit_data' / 'DarkPhoton'
xlim = (1e-11, 5e-4)
ylim = (2e-9, 1e-5)   # floor lowered vs paper Fig. 8 to clear the legend
lab_files = [
    'Coulomb.txt', 'Cavendish.txt', 'PlimptonLawton.txt', 'AFM.txt',
    'Spectroscopy.txt', 'CROWS.txt', 'DarkSRF.txt', 'ALPS.txt',
    'SPring-8.txt', 'LSW_UWA.txt', 'LSW_ADMX.txt',
]
x_grid = np.geomspace(xlim[0], xlim[1], 3000)
env = np.full_like(x_grid, ylim[1])
for fname in lab_files:
    try:
        dat = np.loadtxt(AL / fname)
        m, e = dat[:, 0], dat[:, 1]
        mask = (m >= xlim[0] * 0.01) & (m <= xlim[1] * 100) & np.isfinite(e) & (e > 0)
        if np.sum(mask) < 2:
            continue
        m, e = m[mask], e[mask]
        idx = np.argsort(m)
        m, e = m[idx], e[idx]
        e_interp = 10 ** np.interp(np.log10(x_grid), np.log10(m), np.log10(e),
                                   left=np.log10(ylim[1]), right=np.log10(ylim[1]))
        env = np.minimum(env, e_interp)
    except OSError:
        pass
env = np.clip(env, ylim[0] * 0.1, ylim[1])

# --- Plot ---
fig, ax = plt.subplots(figsize=(DOUBLE_COL, 3.5))

ax.fill_between(x_grid, env, ylim[1], color='0.85', zorder=0, linewidth=0,
                label=r'Existing limits')
ax.plot(x_grid, env, color='0.5', lw=0.8, zorder=1)

ax.loglog(m_pl_v, e_pl_v, color=C['red'], lw=LW_THICK, zorder=5,
          label=r'\texttt{spectroxide} (profile likelihood)')
ax.loglog(m_cj_v, e_cj_v, color=C['orange'], lw=LW_THICK, ls='-.', zorder=6,
          label=r'\texttt{spectroxide} (CCJ24 statistic)')
ax.loglog(cosmotherm_lims[:, 0], cosmotherm_lims[:, 1],
          color=C['blue'], lw=LW_THICK, ls='--', zorder=4,
          label=r'Chluba, Cyr \& Johnson (2024)')
ax.fill_between(m_pl_v, e_pl_v, ylim[1], alpha=0.15, color=C['red'], zorder=0)

ax.set_xlabel(r"Dark photon mass $m_{A'}$ [eV]")
ax.set_ylabel(r'Kinetic mixing $\epsilon$')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.legend(loc='lower left', fontsize=LEGEND_SIZE, framealpha=0.9)

fig.savefig(FIG_DIR / 'dp_firas_pde_constraints_ccj24.pdf',
            bbox_inches='tight', dpi=300)
plt.show()


def pub(m):
    return 10 ** np.interp(np.log10(m), np.log10(cosmotherm_lims[:, 0]),
                           np.log10(cosmotherm_lims[:, 1]))


lo, hi = cosmotherm_lims[:, 0].min(), cosmotherm_lims[:, 0].max()
for label, m, e in (('profile likelihood', m_pl_v, e_pl_v),
                    ('CCJ24 statistic  ', m_cj_v, e_cj_v)):
    k = (m >= lo) & (m <= hi)
    r = e[k] / pub(m[k])
    print(f'{label} / published: median {np.median(r):.3f}, '
          f'range [{r.min():.3f}, {r.max():.3f}]  ({k.sum()} masses)')

## What the comparison shows

- Applying the **CCJ24 statistic** to `spectroxide` templates reproduces the
  published `CosmoTherm` curve with a median ratio of **0.97**
  (range 0.86–1.55, the upper tail confined to the sharp high-mass rise where
  the digitized reference is least reliable). The solver physics is therefore
  validated like-for-like.
- The paper's **profile likelihood** is tighter by a median factor **1.16**
  (ratio 0.84 to the published curve). That offset is a statistics choice —
  full covariance and a floating $T$ rather than a fixed-$T_0$ deprojection with
  diagonal errors — not a difference in the distortion templates, since both
  curves here are fits to the *same* PDE output.